<a href="https://colab.research.google.com/github/Tobi2904/Fine-tune-chatbot/blob/main/TestFineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt Unsloth và các thư viện hỗ trợ
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7r7wpytn/unsloth_16d8f2b685ff4f29b3a5d13ed60b1382
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7r7wpytn/unsloth_16d8f2b685ff4f29b3a5d13ed60b1382
  Resolved https://github.com/unslothai/unsloth.git to commit f7f540a58b853d262ec7ba90c9c0af5e742cc696
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.6 MB/s eta 0:00:00


In [ ]:
from unsloth import FastLanguageModel
import torch

# Khai báo cấu hình
max_seq_length = 2048 # Độ dài tối đa của 1 đoạn hội thoại (tính bằng token)

# Tải Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # Nén 4-bit để vừa với GPU miễn phí
)

# Gắn Adapter (LoRA) vào mô hình
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Kích thước ma trận phụ
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    bias = "none",
     use_gradient_checkpointing = "unsloth",
 )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
import random
from datasets import load_dataset

# 1. TEMPLATE BẮT BUỘC PHẢI CÓ
alpaca_prompt = """Dưới đây là một chỉ thị mô tả một nhiệm vụ, đi kèm với đầu vào cung cấp thêm bối cảnh. Hãy viết một phản hồi hoàn thành yêu cầu một cách chính xác.

### Chỉ thị:
{}

### Đầu vào:
{}

### Phản hồi:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output)
        text = text + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

# 2. CHỈ TẢI DATA CỦA BẠN (Đã xóa dòng yahma thừa)
dataset = load_dataset("json", data_files = "dataset_tuvan_khachhang.json", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. CHIA RANDOM TRAIN/TEST
current_random_seed = random.randint(0, 999999)
split_dataset = dataset.train_test_split(test_size=0.1, seed=current_random_seed, shuffle=True)

train_data = split_dataset["train"]
test_data = split_dataset["test"]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/124 [00:00<?, ? examples/s]

In [ ]:
len(train_data)
len(test_data)

13

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Khởi tạo Cỗ máy huấn luyện
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = test_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,

        # --- CẤU HÌNH TỰ ĐỘNG CHẤM ĐIỂM ---
        eval_strategy = "steps",
        eval_steps = 10,
        logging_first_step = True,

        # 👇 THÊM DÒNG NÀY VÀO ĐỂ FIX LỖI FP16 👇
        optim = "paged_adamw_8bit",

        output_dir = "outputs",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
    ),
)

print("🚀 Bắt đầu quá trình huấn luyện (Training)...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/111 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/13 [00:00<?, ? examples/s]

🚀 Bắt đầu quá trình huấn luyện (Training)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 111 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
10,1.812642,1.538255
20,1.223761,1.260388
30,1.033576,1.163217
40,0.982826,1.136670
50,0.820215,1.111765
60,0.870497,1.114476


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=60, training_loss=1.22544873158137, metrics={'train_runtime': 147.0211, 'train_samples_per_second': 3.265, 'train_steps_per_second': 0.408, 'total_flos': 1564388765589504.0, 'train_loss': 1.22544873158137, 'epoch': 4.285714285714286})

In [ ]:
from transformers import TextStreamer

# 1. BẬT CHẾ ĐỘ SUY LUẬN (INFERENCE)
# Lệnh này cực kỳ quan trọng: Báo cho PyTorch biết "Học xong rồi, giờ chỉ trả lời thôi, tắt hết tính toán đạo hàm đi cho nhẹ RAM"
FastLanguageModel.for_inference(model)

# 2. CHUẨN BỊ CÂU HỎI TEST (ĐÓNG VAI ỨNG VIÊN KHÓ TÍNH)
test_instruction = "Bạn là một chuyên viên tư vấn. Hãy xử lý tình huống sau."
test_input = "Tôi mua hàng về dùng 2 ngày đã hỏng rồi. Chất lượng kém quá!"

# Dùng lại đúng template lúc học, nhưng để trống phần Output {} để AI tự điền
prompt = alpaca_prompt.format(test_instruction, test_input, "")

# 3. MÃ HÓA CÂU HỎI
inputs = tokenizer(
    [prompt],
    return_tensors = "pt"
).to("cuda")

# Cài đặt bộ gõ chữ từ từ (như ChatGPT)
text_streamer = TextStreamer(tokenizer, skip_prompt=True) # skip_prompt=True để màn hình chỉ in câu trả lời, không in lại câu hỏi

print("🤖 AI HR đang trả lời...\n")
print("-" * 50)

# 4. YÊU CẦU AI SINH CÂU TRẢ LỜI
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,  # Giới hạn độ dài câu trả lời (tránh nó nói luyên thuyên không dừng)
    temperature = 0.3,     # Độ sáng tạo: 0.3 là mức lý tưởng cho HR (chuyên nghiệp, điềm đạm, không bịa đặt)
    use_cache = True
)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 AI HR đang trả lời...

--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Dạ, em rất tiếc anh/chị gặp phải tình huống này ạ! Anh/chị đã mua hàng và dùng ngay, không để anh/chị phải chờ đợi. Em xin lỗi vì chất lượng không đáp ứng được kỳ vọng. Anh/chị có thể em hỗ trợ đổi hàng hoặc hoàn tiền không ạ? Em sẽ xử lý ngay để anh/chị không phải mất thêm thời gian. Anh/chị cho em biết cách tiếp tục để em xử lý tốt nhất ạ!<|eot_id|>


In [ ]:
# 1. Lưu trọng số LoRA và bộ mã hóa (Tokenizer) vào thư mục
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 2. Nén thư mục đó lại thành file zip để tải về máy không bị lỗi
!zip -r lora_model.zip lora_model/
print("🎉 Đã nén xong! Hãy vào thư mục bên góc trái Colab để tải file lora_model.zip về máy.")

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


  adding: lora_model/ (stored 0%)
  adding: lora_model/tokenizer_config.json (deflated 96%)
  adding: lora_model/adapter_model.safetensors (deflated 7%)
  adding: lora_model/adapter_config.json (deflated 59%)
  adding: lora_model/README.md (deflated 65%)
  adding: lora_model/chat_template.jinja (deflated 71%)
  adding: lora_model/tokenizer.json (deflated 85%)
🎉 Đã nén xong! Hãy vào thư mục bên góc trái Colab để tải file lora_model.zip về máy.


In [ ]:
from google.colab import files
files.download('lora_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>